<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-25</br>
</div>

</br>

# 학습 내용
>이번 장에서는 <strong>RAG 파이프라인(RAG Pipeline)</strong>에 대해 학습합니다.
>LangGraph StateGraph로 Retrieve → Generate 파이프라인을 학습해봅시다.

</br>

<div style="text-align:center">

</div>

In [12]:
# TODO 0: 환경변수를 로드하고, 이전 실습(001~005)에서 만든 LLM, 임베딩, 문서, 검색기를 준비해봅시다.

import os, glob
from dotenv import load_dotenv
from langchain_upstage import ChatUpstage, UpstageEmbeddings
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

# LLM 초기화 (001)
llm = ChatUpstage(model="solar-pro3", temperature=0)

# 임베딩 모델 초기화 (004)
embeddings = UpstageEmbeddings(model="embedding-query")

# data/ 폴더의 모든 PDF 로드 (002)
all_documents = []
for pdf_path in sorted(glob.glob("data/*.pdf")):
    loader = PyMuPDFLoader(pdf_path)
    docs = loader.load()
    for doc in docs:
        doc.metadata["source_file"] = os.path.basename(pdf_path)
    all_documents.extend(docs)

# 청킹 (005)
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(all_documents)

# 벡터DB + 검색기 (004)
vectorstore = Chroma.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"LLM: {llm.model_name}")
print(f"문서: PDF {len(glob.glob('data/*.pdf'))}개 → {len(all_documents)}페이지 → {len(chunks)}청크")
print(f"검색기 준비 완료")

LLM: solar-pro3
문서: PDF 8개 → 8페이지 → 33청크
검색기 준비 완료


</br>

# RAG (Retrieval-Augmented Generation)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">검색(Retrieval)</mark>으로 관련 문서를 찾고, 이를 컨텍스트로 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">생성(Generation)</mark>하는 파이프라인입니다.

> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">RAG(Retrieval-Augmented Generation, 검색 증강 생성)</mark>는 LLM의 할루시네이션 문제를 외부 지식으로 보완하는 아키텍처입니다. 핵심 아이디어는 단순합니다.</br></br>
> LLM에게 "이 자료를 보고 답해"라고 알려주면, LLM은 자신이 학습한 내용 대신 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">제공된 자료에 근거해 답변</mark>합니다. 이렇게 하면 최신 정보, 조직 내부 문서, 수시로 바뀌는 데이터도 LLM이 정확히 다룰 수 있습니다.</br></br>
> RAG 파이프라인의 전체 흐름은 다음과 같습니다.</br>
> (1) <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">문서 로드</mark>로 PDF, 웹 페이지 등 원본 문서를 `Document` 객체로 변환하고,</br></br>
> (2) <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">청킹</mark>으로 긴 문서를 토큰 제한에 맞는 작은 조각으로 분할합니다.</br></br>
> (3) <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">임베딩</mark>으로 각 청크를 고차원 벡터로 변환한 뒤,</br></br>
> (4) <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">벡터 저장</mark>으로 변환된 벡터를 벡터 데이터베이스(예: Chroma)에 저장합니다.</br></br>
> (5) <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">검색(Retrieve)</mark>에서 사용자 질문을 벡터로 변환하여 가장 유사한 청크를 검색하고,</br></br>
> (6) <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">생성(Generate)</mark>에서 검색된 청크를 컨텍스트로 LLM에 전달하여 최종 답변을 생성합니다.</br></br>
> 이 장에서는 5번과 6번, 즉 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">검색과 생성 단계를 LangGraph로 연결하는 방법</mark>을 학습합니다.</br>

</br>

## RAGState 정의
> StateGraph의 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">상태 타입</mark>을 TypedDict로 정의합니다.

In [13]:
# TODO 1: 상태 타입을 정의하여 question(str), context(List[Document]), answer(str) 필드를 선언한 뒤 필드 목록을 출력해봅시다.

from typing import TypedDict, List
from langchain_core.documents import Document

class RAGState(TypedDict):
    question: str
    context: List[Document]
    answer: str

print(f"RAGState 필드: {list(RAGState.__annotations__.keys())}")

RAGState 필드: ['question', 'context', 'answer']


</br>

## 노드 함수 구현
> 각 노드는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">상태를 받아 상태를 반환</mark>하는 함수입니다.

In [14]:
# TODO 2: retrieve 함수를 정의하세요. RAGState를 받아 retriever로 관련 문서를 검색하고 context를 반환하세요.

def retrieve(state: RAGState) -> RAGState:
    """질문으로 관련 문서 검색"""
    documents = retriever.invoke(state["question"])
    return {"context": documents}

In [15]:
# TODO 3: generate 함수를 정의하세요. RAGState를 받아 프롬프트 템플릿과 LLM으로 답변을 생성하고 answer를 반환하세요.

def generate(state: RAGState) -> RAGState:
    """검색된 문서를 기반으로 답변 생성"""
    context = "\n".join([doc.page_content for doc in state["context"]])
    prompt = ChatPromptTemplate.from_template("""
    다음 자료를 기반으로 답변하세요.
    자료: {context}
    질문: {question}
    """)
    chain = prompt | llm
    response = chain.invoke({
        "context": context,
        "question": state["question"]
    })
    return {"answer": response.content}

</br>

## StateGraph 조립
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">노드를 추가하고 엣지로 연결</mark>하여 그래프를 완성합니다.

In [16]:
# TODO 4: 상태 그래프를 생성하고, retrieve와 generate 노드를 추가한 뒤 START→retrieve→generate→END로 엣지를 연결하여 컴파일 후 "총알배송 서비스의 혜택은?"으로 실행해봅시다.

from langgraph.graph import StateGraph, START, END

graph = StateGraph(RAGState)

# 노드 추가
graph.add_node("retrieve", retrieve)
graph.add_node("generate", generate)

# 엣지 연결
graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", END)

# 컴파일 및 실행
app = graph.compile()
result = app.invoke({"question": "총알배송 서비스의 혜택은?"})
print(f"질문: {result['question']}")
print(f"검색 문서: {len(result['context'])}건")
print(f"답변: {result['answer']}")

질문: 총알배송 서비스의 혜택은?
검색 문서: 3건
답변: 총알배송 서비스의 혜택은 다음과 같습니다:

1. **아침배송**:  
   - 표시 상품을 **오늘 밤 10시 전 주문**하면 **내일 아침 7시 전**에 배송받을 수 있습니다.  
   - 단, **관공서, 학교, 기숙사, 병원** 등은 출입이 제한되어 아침배송이 불가능할 수 있습니다.

2. **배송 지역**:  
   - **서울, 수도권, 충청권**  
   - **호남권**: 광주, 군산, 익산, 전주  
   - **영남권**: 부산, 대구, 울산, 창원, 김해, 양산  

3. **주문 마감 시간**:  
   - **서울, 수도권(평택 제외), 인천**: 토요일 21시까지 주문 → 월요일 또는 화요일 배송  
   - **그 외 지역**: 토요일 18시까지 주문 → 월요일 또는 화요일 배송  

4. **배송비**:  
   - 15,000원 이상 구매 시 **무료**  
   - 15,000원 미만 주문 시 **2,500원**  

5. **배송 상품**:  
   - 일부 상품은 제외되며, 상세페이지에서 아침배송 가능 여부를 확인해야 합니다.  

※ 물류센터 사정에 따라 배송 일정이 일부 조정될 수 있습니다.


</br>

## RAG 파이프라인 구성요소 정리

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">구성요소</th>
      <th>역할</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center"><code>RAGState</code></td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">상태 스키마</mark> 정의</td></tr>
    <tr><td style="text-align:center"><code>retrieve</code></td><td>벡터 검색 → context 업데이트</td></tr>
    <tr><td style="text-align:center"><code>generate</code></td><td>context 기반 → answer 생성</td></tr>
    <tr><td style="text-align:center"><code>StateGraph</code></td><td>노드와 엣지로 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">워크플로우 정의</mark></td></tr>
    <tr><td style="text-align:center"><code>compile()</code></td><td>실행 가능한 앱으로 변환</td></tr>
  </tbody>
</table>

💡StateGraph의 장점
> 단순 체인(`prompt | llm`)보다 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">분기, 반복, 조건부 실행</mark>이 가능합니다.
> Agent 확장 시 동일한 패턴으로 복잡한 워크플로우를 구성할 수 있습니다.

💡RAG vs Fine-tuning
> RAG: <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">외부 지식을 실시간</mark>으로 주입 (비용 낮음, 최신성 보장)
> Fine-tuning: 모델 가중치 자체를 변경 (비용 높음, 도메인 특화)